In [ ]:
import pandas as pd
import re

# =============================
# LOAD DATA
# =============================
file_path = "input.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master")

# =============================
# CLEAN CYCLE TIME
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# Cycle time in minutes
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
BASE_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
EXTRA_MACHINE = "TOYO 80T 2ND"
ALL_TARGET_MACHINES = BASE_120T_MACHINES | {EXTRA_MACHINE}

AVAILABLE_TIME_MIN = 22 * 60  # 1320 minutes

# =============================
# HELPER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\- ]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# BUILD PART → MACHINE MAP
# =============================
records = []

for child, grp in master.groupby("Child Part", sort=False):

    raw = ",".join(grp["Vertical Machines"].astype(str))
    tokens = re.split(r"[,\|/\\\n]+", raw)

    machines = set()
    for t in tokens:
        m = normalize_machine(t)
        if m in BASE_120T_MACHINES:
            machines.add(m)

    # 🔥 If part is 120T-capable, add TOYO 80T 2ND
    if machines:
        machines.add(EXTRA_MACHINE)

    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Possible Machines": ", ".join(sorted(machines))
    })

part_machine_df = pd.DataFrame(records)

# =============================
# BUILD PART-WISE CAPACITY TABLE
# =============================
capacity_rows = []

for _, row in part_machine_df.iterrows():
    child = row["Child Part"]
    ct = cycle_time_min.get(child)

    if not ct or ct <= 0:
        continue

    capacity = AVAILABLE_TIME_MIN / ct

    for machine in row["Possible Machines"].split(", "):
        capacity_rows.append({
            "Child Part": child,
            "Machine": machine,
            "Capacity (units/day)": int(capacity)
        })

capacity_df = pd.DataFrame(capacity_rows)

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== PART → POSSIBLE MACHINES ==========\n")
display(part_machine_df)

print("\n========== PART-WISE MACHINE CAPACITY (1 DAY) ==========\n")
display(capacity_df)
